In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import ResNet50
import tensorflow_datasets as tfds

IMG_SIZE = 224
BATCH_SIZE = 32

dataset, info = tfds.load('deep_weeds', split='train', with_info=True, as_supervised=True)


In [3]:
# Keep only labels 0-4
dataset = dataset.filter(lambda image, label: label < 5)

In [4]:
# Use info to get number of examples
num_examples = info.splits['train'].num_examples

train_size = int(0.8 * num_examples)

# Take 80% for training, skip 80% for validation
train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size)


In [5]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def preprocess(image, label):
    # Resize
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    # Preprocess for ResNet50 (-1..1)
    image = preprocess_input(image)
    return image, label


In [6]:
train_dataset = train_dataset.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [7]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False 

model = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,769,413 (94.49 MB)

 Trainable params: 1,181,701 (4.51 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [8]:
# fit the model
history = model.fit(train_dataset, validation_data=val_dataset, epochs=10)

Epoch 1/10
     51/Unknown 26s 120ms/step - accuracy: 0.4756 - loss: 2.5692

KeyboardInterrupt: 

In [9]:

for layer in base_model.layers[-40:]:
    layer.trainable = True


model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,769,413 (94.49 MB)

 Trainable params: 17,013,509 (64.90 MB)

 Non-trainable params: 7,755,904 (29.59 MB)

In [10]:
# fit the model
history = model.fit(train_dataset, validation_data=val_dataset, epochs=10)

Epoch 1/10
    166/Unknown 69s 250ms/step - accuracy: 0.7845 - loss: 1.3008

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


166/166 ━━━━━━━━━━━━━━━━━━━━ 80s 316ms/step - accuracy: 0.7848 - loss: 1.2993
Epoch 2/10
166/166 ━━━━━━━━━━━━━━━━━━━━ 40s 217ms/step - accuracy: 0.9215 - loss: 0.6903
Epoch 3/10
166/166 ━━━━━━━━━━━━━━━━━━━━ 41s 227ms/step - accuracy: 0.9595 - loss: 0.4300
Epoch 4/10
166/166 ━━━━━━━━━━━━━━━━━━━━ 40s 220ms/step - accuracy: 0.9765 - loss: 0.3037
Epoch 5/10
166/166 ━━━━━━━━━━━━━━━━━━━━ 40s 222ms/step - accuracy: 0.9772 - loss: 0.2860
Epoch 6/10
166/166 ━━━━━━━━━━━━━━━━━━━━ 62s 345ms/step - accuracy: 0.9854 - loss: 0.2048
Epoch 7/10
166/166 ━━━━━━━━━━━━━━━━━━━━ 42s 229ms/step - accuracy: 0.9800 - loss: 0.1889
Epoch 8/10
166/166 ━━━━━━━━━━━━━━━━━━━━ 47s 265ms/step - accuracy: 0.9920 - loss: 0.1348
Epoch 9/10
166/166 ━━━━━━━━━━━━━━━━━━━━ 48s 268ms/step - accuracy: 0.9917 - loss: 0.1205
Epoch 10/10
 11/166 ━━━━━━━━━━━━━━━━━━━━ 45s 294ms/step - accuracy: 0.9806 - loss: 0.1397

: 

: 

: 